In [ ]:
import os
import pandas as pd

# 1. Read Artifact from Notebook 1
input_artifact = "artifacts/01_raw_joined_orders.csv"
print(f"Loading artifact from: {input_artifact}")
df = pd.read_csv(input_artifact)

print(f"Initial Shape: {df.shape}")

# 2. Filter for delivered orders with valid delivery dates
df_delivered = df[(df['order_status'] == 'delivered') & (df['order_delivered_customer_date'].notna())].copy()
print(f"Shape after filtering delivered orders: {df_delivered.shape}")

# 3. Convert date columns to datetime objects
df_delivered['order_delivered_customer_date'] = pd.to_datetime(df_delivered['order_delivered_customer_date'])
df_delivered['order_estimated_delivery_date'] = pd.to_datetime(df_delivered['order_estimated_delivery_date'])

# 4. Create the target label: is_late
# 1 if actual delivery date is strictly greater than estimated delivery date, else 0
df_delivered['is_late'] = (df_delivered['order_delivered_customer_date'] > df_delivered['order_estimated_delivery_date']).astype(int)

# 5. Sanity Check on real orders
print("\nSample check of dates vs label:")
sample_check = df_delivered[['order_delivered_customer_date', 'order_estimated_delivery_date', 'is_late']].head(5)
print(sample_check)

# 6. Analyze Class Imbalance
counts = df_delivered['is_late'].value_counts()
percentages = df_delivered['is_late'].value_counts(normalize=True) * 100

print("\n--- Class Distribution ---")
print(f"On-Time (0): {counts.get(0, 0)} orders ({percentages.get(0, 0):.2f}%)")
print(f"Late    (1): {counts.get(1, 0)} orders ({percentages.get(1, 0):.2f}%)")

output_artifact = "artifacts/02_labeled_orders.csv"
df_delivered.to_csv(output_artifact, index=False)
print(f"\nArtifact successfully saved to: {output_artifact}")

Loading artifact from: artifacts/01_raw_joined_orders.csv
Initial Shape: (99441, 18)
Shape after filtering delivered orders: (96470, 18)

Sample check of dates vs label:
  order_delivered_customer_date order_estimated_delivery_date  is_late
0           2017-10-10 21:25:13                    2017-10-18        0
1           2018-08-07 15:27:45                    2018-08-13        0
2           2018-08-17 18:06:29                    2018-09-04        0
3           2017-12-02 00:28:42                    2017-12-15        0
4           2018-02-16 18:17:02                    2018-02-26        0

--- Class Distribution ---
On-Time (0): 88644 orders (91.89%)
Late    (1): 7826 orders (8.11%)

Artifact successfully saved to: artifacts/02_labeled_orders.csv
